In [2]:
import pandas as pd

# TSV 로드
df = pd.read_csv(r"C:\Users\Kunny\Research\Dataset\Missense Variant dataset\rhapsody2_sav_db.tsv", sep="\t", header=None)
df.columns = ["UniProtID", "StructureFile", "MutPos", "WT", "Mut", "Label"]

In [ ]:
print(df)

,UniProtID,StructureFile,MutPos,WT,Mut,Label,InSwissProt
0,A0A087WXS9,AF-A0A087WXS9-F1-model_v4.pdb,69,R,L,0,True
1,A0A096LP55,AF-A0A096LP55-F1-model_v4.pdb,53,Y,C,0,True
2,A0A0A6YYG8,AF-A0A0A6YYG8-F1-model_v4.pdb,112,G,R,0,False
3,A0A0A6YYL4,AF-A0A0A6YYL4-F1-model_v4.pdb,193,R,Q,0,False
4,A0A0J9YWL9,AF-A0A0J9YWL9-F1-model_v4.pdb,203,P,T,0,True
...,...,...,...,...,...,...,...
117520,Q9Y6Y8,AF-Q9Y6Y8-F1-model_v4.pdb,45,A,T,0,True
117521,Q9Y6Y8,AF-Q9Y6Y8-F1-model_v4.pdb,718,A,V,0,True
117522,Q9Y6Z7,AF-Q9Y6Z7-F1-model_v4.pdb,179,R,W,0,True
117523,Q9Y6Z7,AF-Q9Y6Z7-F1-model_v4.pdb,36,A,T,0,True


In [50]:
uniprot_ids = set(df["UniProtID"].tolist())
len(uniprot_ids)

12094

In [51]:
print(df[df["UniProtID"]=="A0A087WXS9"])

    UniProtID                  StructureFile  MutPos WT Mut  Label
0  A0A087WXS9  AF-A0A087WXS9-F1-model_v4.pdb      69  R   L      0


In [4]:
from Bio import SeqIO

# 2. FASTA에서 UniProt ID만 추출 (앞부분만 파싱)
fasta_path = r"C:\Users\Kunny\Research\Dataset\Missense Variant dataset\uniprot_sprot.fasta"
available_ids = set()

for record in SeqIO.parse(fasta_path, "fasta"):
    # FASTA header: >sp|A0A087WXS9|... 또는 >tr|... 이므로 split("|")[1]로 ID 추출
    try:
        entry_id = record.id.split("|")[1]
        available_ids.add(entry_id)
    except IndexError:
        continue

# 3. 존재 여부 확인
df["InSwissProt"] = df["UniProtID"].isin(available_ids)

# 4. 결과 요약
print(df["InSwissProt"].value_counts())


InSwissProt
True     117523
False         2
Name: count, dtype: int64


In [ ]:
df[~df["InSwissProt"]]["UniProtID"].unique()

array(['A0A0A6YYG8', 'A0A0A6YYL4'], dtype=object)

In [6]:
from Bio import SeqIO

# UniProt ID → 시퀀스 딕셔너리 만들기
fasta_path = r"C:\Users\Kunny\Research\Dataset\Missense Variant dataset\uniprot_sprot.fasta"
id_to_seq = {}

for record in SeqIO.parse(fasta_path, "fasta"):
    uniprot_id = record.id.split('|')[1]  # 예: sp|P12345| -> P12345
    id_to_seq[uniprot_id] = str(record.seq)

# 일치 여부 확인 예시 (처음 10개만)
for i, row in df[df["InSwissProt"]].head(10000).iterrows():
    uid = row["UniProtID"]
    pos = int(row["MutPos"]) - 1  # Python은 0-based index
    wt = row["WT"]

    seq = id_to_seq.get(uid)
    if seq and pos < len(seq):
        actual = seq[pos]
        print(f"{uid} at position {pos+1}: expected={wt}, actual={actual} →", "✅ Match" if actual == wt else "❌ MISMATCH")
    else:
        print(f"{uid}: Sequence not found or position out of range")


A0A087WXS9 at position 69: expected=R, actual=R → ✅ Match
A0A096LP55 at position 53: expected=Y, actual=Y → ✅ Match
A0A0J9YWL9 at position 203: expected=P, actual=P → ✅ Match
A0A0J9YWL9 at position 39: expected=R, actual=R → ✅ Match
A0A0J9YWL9 at position 509: expected=L, actual=L → ✅ Match
A0A0J9YWL9 at position 515: expected=L, actual=L → ✅ Match
A0A0J9YWL9 at position 518: expected=L, actual=L → ✅ Match
A0A0J9YWL9 at position 855: expected=S, actual=S → ✅ Match
A0A0J9YY54 at position 261: expected=P, actual=P → ✅ Match
A0A0J9YY54 at position 326: expected=G, actual=G → ✅ Match
A0A0J9YY54 at position 383: expected=M, actual=M → ✅ Match
A0A0U1RQS6 at position 172: expected=R, actual=R → ✅ Match
A0A1B0GWG4 at position 59: expected=R, actual=R → ✅ Match
A0AUZ9 at position 202: expected=P, actual=P → ✅ Match
A0AUZ9 at position 429: expected=M, actual=M → ✅ Match
A0AUZ9 at position 84: expected=R, actual=R → ✅ Match
A0AUZ9 at position 968: expected=D, actual=D → ✅ Match
A0AV02 at position

In [66]:
id_to_seq["A0A0J9YY54"]

'MAMNFGDHASGFRHNDVIRFINNEVLMDGSGPAFYVAFRSRPWNEVEDSLQAIVADSQVPRAIKRACTWSALALSVRVATRQREELLHHVRRLQRHAEERQATSWALTSQLQQLRLEHEVAATQLHLAQAALQQALNERDGLYGRLLQIERFPQAAPLAHEIMSGPQAEQNGAAACPLATEQQSDMVAMGTHANAQMPTPTDVLYVPGPLSPWAQGMQPPLPVPHPFPHPPPFPMKFPSLPPLPPAVVTGAEAAAVPLQMPPTEIHPPCPWPAVGFQEEMAPLWYQRSYIQEEDSKILQGSFPLGDSRSHSQGEGSERSQRMPLPGDSGCHNPLSESPQGTAPLGSSGCHSQEEGTEGPQGMDPLGNRERQNQKEGPKRARRMHTLVFRRSHKSEGPEGPQGTVPQGDSRSYSQEGCSDRAQEMATLVFIRRCKPEGPKRPQWTVPLGDSRSHIKEEGPEGPQRIVLQGDNRSYSQEGSPERAQGMATLVFSRSCKPEEGPERPQDTPLGDSRSHIKEEGPEGPQRIVLQGDNRSYSQEGSPERAQGMATLVFSRSCKPEEGPERPQDTPLGDSRSHIKEEGPEGPQRIVLQGDNRSYSQEGSRERAQGMATLVFSRSCKPEEGPERPQGTPLGDSRSHGVRESPKKWQPQRQKAKKPKVNKVSGSQQQEKPASFPVPVNWKCPWCKAINFSWRTACYKCKKACVPFESGGQTQ'

In [47]:
id_to_seq["A0A096LP55"][52]

'Y'

In [ ]:
from Bio import SeqIO

# 2. FASTA에서 UniProt ID만 추출 (앞부분만 파싱)
fasta_path = r"C:\Users\Kunny\Research\Dataset\Missense Variant dataset\uniprot_sprot.fasta"
available_ids = set()

for record in SeqIO.parse(fasta_path, "fasta"):
    # FASTA header: >sp|A0A087WXS9|... 또는 >tr|... 이므로 split("|")[1]로 ID 추출
    try:
        entry_id = record.id.split("|")[1]
        available_ids.add(entry_id)
    except IndexError:
        continue

# 3. 존재 여부 확인
df["InSwissProt"] = df["UniProtID"].isin(available_ids)

# 4. 결과 요약
print(df["InSwissProt"].value_counts())


InSwissProt
True     117523
False         2
Name: count, dtype: int64


In [48]:
import requests

# 다운로드할 URL
url = "https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/uniprot_trembl.fasta.gz"

# 저장할 경로 (원하는 경로로 변경하세요)
save_path = "E:/CAGI_data/uniprot_trembl.fasta.gz"  # 예: "C:/Users/YourName/Downloads/uniprot_trembl.fasta.gz"

# 파일 다운로드 및 저장
response = requests.get(url, stream=True)
response.raise_for_status()  # 오류가 발생하면 예외 발생

with open(save_path, "wb") as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)

print(f"✅ 다운로드 완료: {save_path}")

✅ 다운로드 완료: E:/CAGI_data/uniprot_trembl.fasta.gz


In [ ]:
import pandas as pd
from Bio import SeqIO
import gzip

# 파일 경로
sprot_path = r"C:\Users\Kunny\Research\Dataset\Missense Variant dataset\uniprot_sprot.fasta"
trembl_path = r"E:\CAGI_data\uniprot_trembl.fasta.gz"
tsv_path = r"C:\Users\Kunny\Research\Dataset\Missense Variant dataset\rhapsody2_sav_db.tsv"

# 1. Swiss-Prot FASTA → 딕셔너리
id_to_seq = {}
for record in SeqIO.parse(sprot_path, "fasta"):
    uniprot_id = record.id.split("|")[1]  # sp|P12345| -> P12345
    id_to_seq[uniprot_id] = str(record.seq)

# 2. Rhapsody2 TSV 로드
df = pd.read_csv(tsv_path, sep="\t", header=None)
df.columns = ["UniProtID", "StructureFile", "MutPos", "WT", "Mut", "Label"]

# 3. 못 찾은 UniProt ID 목록
uniprot_ids = set(df["UniProtID"])
missing_ids = [uid for uid in uniprot_ids if uid not in id_to_seq]

print(f"Swiss-Prot에서 찾지 못한 ID 수: {len(missing_ids)}")

# 4. TrEMBL에서 못 찾은 ID들 찾아서 딕셔너리에 추가
found_in_trembl = 0
with gzip.open(trembl_path, "rt") as handle:
    for record in SeqIO.parse(handle, "fasta"):
        uniprot_id = record.id.split("|")[1]
        if uniprot_id in missing_ids:
            id_to_seq[uniprot_id] = str(record.seq)
            found_in_trembl += 1
            if found_in_trembl == len(missing_ids):
                break  # 다 찾았으면 중단

print(f"TrEMBL에서 추가로 찾은 시퀀스 수: {found_in_trembl}")
print(f"최종 UniProtID → 시퀀스 딕셔너리 크기: {len(id_to_seq)}")

import json
with open(r"C:\Users\Kunny\Research\Dataset\Missense Variant dataset\UniProtID_to_seq.json", "w") as f:
    json.dump(id_to_seq, f)


Swiss-Prot에서 찾지 못한 ID 수: 2
TrEMBL에서 추가로 찾은 시퀀스 수: 2
최종 UniProtID → 시퀀스 딕셔너리 크기: 573663


In [17]:
import json
import os
import pandas as pd

# 경로 설정
json_path = r"C:\Users\Kunny\Research\Dataset\Missense Variant dataset\UniProtID_to_seq.json"
tsv_path = r"C:\Users\Kunny\Research\Dataset\Missense Variant dataset\rhapsody2_sav_db.tsv"
output_dir = r"E:\CAGI_data\fasta_files"
os.makedirs(output_dir, exist_ok=True)

# JSON 불러오기
with open(json_path, "r") as f:
    id_to_seq = json.load(f)

# TSV 불러오기
df = pd.read_csv(tsv_path, sep="\t", header=None)
df.columns = ["UniProtID", "StructureFile", "MutPos", "WT", "Mut", "Label"]

# 필요한 ID 추출
unique_ids = df["UniProtID"].unique()

# 저장 및 매핑되지 않은 ID 추적
unmatched_ids = []

for uid in unique_ids:
    if uid in id_to_seq:
        fasta_path = os.path.join(output_dir, f"{uid}.fasta")
        with open(fasta_path, "w") as f:
            f.write(f">{uid}\n{id_to_seq[uid]}\n")
    else:
        unmatched_ids.append(uid)

# 매핑 안 된 ID 출력
if unmatched_ids:
    print(f"\n[❗] 다음 UniProt ID는 JSON에 매핑되지 않았습니다 ({len(unmatched_ids)}개):")
    for uid in unmatched_ids:
        print(uid)
else:
    print("\n✅ 모든 UniProt ID가 JSON에 매핑되었습니다.")


✅ 모든 UniProt ID가 JSON에 매핑되었습니다.


In [ ]:
import os
import subprocess

# DIAMOND 실행 경로 및 입력 디렉토리 설정
diamond_path = r"C:\Users\Kunny\Desktop\정경건\tools\diamond.exe"
db_path = r"E:\CAGI_data\uniref90"
query_dir = r"E:\CAGI_data\fasta_files"
output_dir = r"E:\CAGI_data\blast_results"
os.makedirs(output_dir, exist_ok=True)

# 모든 FASTA에 대해 BLAST 수행
for fname in os.listdir(query_dir):
    if not fname.endswith(".fasta"):
        continue

    uid = fname[:-6]  # .fasta 제거
    query_path = os.path.join(query_dir, fname)
    blast_path = os.path.join(output_dir, f"{uid}.m8")

    if os.path.exists(blast_path):  # 이미 있으면 스킵
        continue

    cmd = [
        diamond_path, "blastp",
        "--db", db_path,
        "--query", query_path,
        "--out", blast_path,
        "--outfmt", "6",
        "--evalue", "0.001",
        "--max-target-seqs", "50000",
        "--threads", "12"  # 원하면 조정
    ]

    print(f"[{uid}] Running DIAMOND blastp...")
    subprocess.run(cmd)

[A0A087WXS9] Running DIAMOND blastp...


In [ ]:
from multiprocessing import Pool
from functools import partial
import os

# DIAMOND 실행 경로 및 입력 디렉토리 설정
diamond_path = r"C:\Users\Kunny\Desktop\정경건\tools\diamond.exe"
db_path = r"E:\CAGI_data\uniref90"
query_dir = r"E:\CAGI_data\fasta_files"
output_dir = r"E:\CAGI_data\blast_results"
os.makedirs(output_dir, exist_ok=True)

def run_diamond(uid, query_path, blast_path):
    cmd = [
        diamond_path, "blastp",
        "--db", db_path,
        "--query", query_path,
        "--out", blast_path,
        "--outfmt", "6",
        "--evalue", "0.001",
        "--max-target-seqs", "50000",
        "--threads", "4"  
    ]
    print(f"[{uid}] Running DIAMOND...")
    subprocess.run(cmd)

# 경로 세팅
tasks = []
for fname in os.listdir(query_dir):
    if fname.endswith(".fasta"):
        uid = fname[:-6]
        query_path = os.path.join(query_dir, fname)
        blast_path = os.path.join(output_dir, f"{uid}.m8")
        if not os.path.exists(blast_path):
            tasks.append((uid, query_path, blast_path))

# 병렬 실행
with Pool(processes=6) as pool:  
    pool.starmap(run_diamond, tasks)

In [13]:
id_to_seq["A0A0J9YY54"][58]

'V'

In [14]:
import json
with open(r"C:\Users\Kunny\Research\Dataset\Missense Variant dataset\UniProtID_to_seq.json", "w") as f:
    json.dump(id_to_seq, f)